[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremylongshore/claude-code-plugins-plus-skills/blob/main/tutorials/skills/03-build-your-first-skill.ipynb)

# Build Your First Skill: Hands-On Tutorial

**Learning Path**: Skills → Plugins → Orchestration  
**Level**: Intermediate  
**Time**: 45 minutes  
**Prerequisites**: [01-what-is-skill.ipynb](01-what-is-skill.ipynb), [02-skill-anatomy.ipynb](02-skill-anatomy.ipynb)

---

## What You'll Build

A production-ready skill: **`test-file-generator`**

**What it does**: Generates test files for code files (Python, JS, Go, etc.)

**Why this skill**:
- ✅ Practical - You'll actually use it
- ✅ Demonstrates core patterns
- ✅ Shows tool authorization
- ✅ Includes validation
- ✅ Enterprise compliant

**By the end, you'll have**:
- A complete SKILL.md file
- Validated against enterprise standards
- Ready to install and use

---

## Step 1: Plan Your Skill

Before writing code, answer these questions:

In [ ]:
# Skill Planning Worksheet
skill_plan = {
    "name": "test-file-generator",
    
    "what": "Generate test files for code files",
    
    "when": [
        "User needs tests for a file",
        "TDD workflow (tests first)",
        "Coverage improvement"
    ],
    
    "inputs": [
        "Source file path",
        "Programming language (detected or specified)",
        "Test framework (Jest, pytest, etc.)"
    ],
    
    "outputs": [
        "Test file with boilerplate",
        "Test cases for public functions/classes",
        "Imports and setup code"
    ],
    
    "tools_needed": [
        "Read (to analyze source file)",
        "Write (to create test file)",
        "Grep (to find functions/classes)"
    ],
    
    "trigger_phrases": [
        "generate test",
        "create test file",
        "write tests for",
        "test coverage"
    ]
}

print("SKILL PLANNING WORKSHEET")
print("=" * 60)
for key, value in skill_plan.items():
    print(f"\n{key.upper().replace('_', ' ')}:")
    if isinstance(value, list):
        for item in value:
            print(f"  • {item}")
    else:
        print(f"  {value}")

print("\n" + "=" * 60)
print("✅ Planning complete - ready to build!")

---

## Step 2: Write the Frontmatter

Using our planning worksheet, let's build the YAML frontmatter:

In [ ]:
import re

def create_skill_frontmatter(name, description, tools, version="1.0.0",
                            license="MIT", author="Your Name <your@email.com>",
                            tags=None, model=None, compatibility=None,
                            compatible_with=None):
    """Create skill frontmatter compliant with the Intent Solutions standard.
    
    Required fields: name, description, allowed-tools, version, author, license
    Optional fields: model, context, agent, user-invocable, argument-hint,
                     hooks, compatibility, compatible-with, tags
    """
    
    # Validate name
    if not re.match(r'^[a-z0-9-]+$', name):
        raise ValueError(f"Name must be kebab-case: {name}")
    
    # Build frontmatter
    lines = [
        "---",
        f"name: {name}",
        "description: |",
    ]
    
    # Handle multiline description
    for line in description.strip().split('\n'):
        lines.append(f"  {line}")
    
    # Required fields
    lines.extend([
        f'allowed-tools: "{tools}"',
        f"version: {version}",
        f"author: {author}",
        f"license: {license}",
    ])
    
    # Optional fields
    if tags:
        lines.append(f"tags: {tags}")
    
    if model:
        # Valid model values: sonnet, haiku, opus
        lines.append(f"model: {model}")
    
    if compatibility:
        lines.append(f'compatibility: "{compatibility}"')
    
    if compatible_with:
        lines.append(f"compatible-with: {compatible_with}")
    
    lines.append("---")
    
    return "\n".join(lines)

# Create frontmatter for our test-file-generator skill
# Note: Bash is always scoped — Bash(npm:*), Bash(git:*), never bare Bash
frontmatter = create_skill_frontmatter(
    name="test-file-generator",
    description="""Generate test files for source code files with framework-specific boilerplate.
Use when: creating tests, improving coverage, TDD workflow.
Triggers: generate test, create test file, write tests, test coverage.""",
    tools="Read,Write,Grep,Bash(npm:*)",
    author="Claude Developer <dev@example.com>",
    tags=["testing", "tdd", "coverage", "productivity"],
    model="sonnet",  # Valid values: sonnet, haiku, opus
    compatible_with="claude-code"
)

print("GENERATED FRONTMATTER:")
print("=" * 60)
print(frontmatter)
print("=" * 60)
print("\nRequired fields: name, description, allowed-tools, version, author, license")
print("Optional fields: model, context, agent, user-invocable, argument-hint,")
print("                 hooks, compatibility, compatible-with, tags")
print("\n\u2705 Frontmatter created!")

---

## Step 3: Write the Body

Now create the body with the 7 required sections: Overview, Prerequisites, Instructions, Output, Error Handling, Examples, Resources.

In [ ]:
# Create the skill body using the 7 required sections:
# Overview, Prerequisites, Instructions, Output, Error Handling, Examples, Resources
#
# Target: ≤150 lines (500 max). Heavy content goes in references/implementation.md
# (Progressive Disclosure Architecture — PDA)

body = """# Test File Generator

Generate test files for source code with appropriate boilerplate, imports, and test case stubs based on detected language and test framework.

## Overview

This skill analyzes a source file to identify public functions, classes, and exports, then generates a complete test file with the correct framework boilerplate and test stubs. It supports Python (pytest/unittest), JavaScript/TypeScript (Jest/Vitest/Mocha), and Go (testing).

## Prerequisites

- Source file must exist and be readable
- Project should have a test framework installed (pytest, Jest, Vitest, etc.)
- Test directory structure should follow project conventions

## Instructions

### Step 1: Analyze Source File

Use the **Read** tool to read the source file and detect:
1. Programming language (from file extension)
2. Public functions, classes, and exported modules

### Step 2: Determine Test Framework

Use **Grep** to detect the project's test framework:
- **Python**: look for `pytest.ini`, `conftest.py`, or `setup.cfg [tool:pytest]`; fallback to `unittest`
- **JS/TS**: look for `jest.config.*`, `vitest.config.*`, or `.mocharc.*`
- **Go**: use the standard `testing` package

### Step 3: Determine Test File Path

Follow project conventions:
- **Python**: `src/module.py` -> `tests/test_module.py`
- **JS/TS**: `src/module.ts` -> `src/module.test.ts`
- **Go**: `module.go` -> `module_test.go` (same directory)

### Step 4: Generate and Write Test File

Use **Write** to create the test file with:
- Framework imports and source module imports
- Setup/teardown scaffolding if needed
- One test stub per public function/class with TODO markers

For detailed templates per language, see `${CLAUDE_SKILL_DIR}/references/implementation.md`.

## Output

Return a summary:
```
Test file created: tests/test_mymodule.py

Generated 3 test stubs:
- test_function_one
- test_function_two
- test_MyClass

Next steps:
1. Implement test cases (search for TODO)
2. Run tests: pytest tests/test_mymodule.py
3. Add edge cases as needed
```

## Error Handling

| Scenario | Action |
|----------|--------|
| Source file not found | Return error with path suggestion |
| Test file already exists | Ask user whether to overwrite or append |
| Cannot detect language | Ask user to specify language or extension |
| No test framework detected | Use language defaults and suggest installing a framework |

## Examples

**Example 1 — Python file**
```
User: "Generate tests for src/calculator.py"
1. Reads src/calculator.py → finds add(), subtract(), multiply()
2. Detects pytest (from pytest.ini)
3. Creates tests/test_calculator.py with 3 test stubs
4. Returns summary with run command
```

**Example 2 — TypeScript file**
```
User: "Create tests for src/utils/parser.ts"
1. Reads src/utils/parser.ts → finds exported functions and classes
2. Detects Vitest (from vitest.config.ts)
3. Creates src/utils/parser.test.ts
4. Returns summary with test command
```

## Resources

- [pytest documentation](https://docs.pytest.org/)
- [Jest documentation](https://jestjs.io/docs/getting-started)
- [Vitest documentation](https://vitest.dev/guide/)
- [Go testing package](https://pkg.go.dev/testing)
- See `${CLAUDE_SKILL_DIR}/references/implementation.md` for full language templates
"""

print("SKILL BODY:")
print("=" * 60)
print(body[:500] + "...\n(truncated for display)")
print("=" * 60)

line_count = len(body.strip().split('\n'))
word_count = len(body.split())
print(f"\nBody stats:")
print(f"  Lines: {line_count}  (target: ≤150, max: 500)")
print(f"  Words: {word_count}")
print(f"  Characters: {len(body)}")

if line_count <= 150:
    print(f"\n✅ Within 150-line target!")
elif line_count <= 500:
    print(f"\n⚠️  Over 150-line target but within 500 max.")
    print("  Consider moving content to references/implementation.md (PDA)")
else:
    print(f"\n❌ Over 500-line maximum! Must move content to references/")

print("\nRequired sections present:")
for section in ["Overview", "Prerequisites", "Instructions", "Output",
                "Error Handling", "Examples", "Resources"]:
    marker = "✅" if f"## {section}" in body else "❌"
    print(f"  {marker} {section}")
print("\n✅ Body created!")

---

## Step 4: Combine Into Complete SKILL.md

In [ ]:
# Combine frontmatter + body
complete_skill = f"{frontmatter}\n\n{body}"

print("COMPLETE SKILL.MD:")
print("=" * 60)
print(complete_skill[:1000])
print("\n...")
print(complete_skill[-500:])
print("=" * 60)
print(f"\nTotal length: {len(complete_skill)} characters")
print("✅ Complete skill assembled!")

---

## Step 5: Validate Against the Skill Standard

In [ ]:
def validate_skill_complete(skill_content):
    """Complete skill validation (Intent Solutions standard)"""
    errors = []
    warnings = []
    
    # Parse frontmatter and body
    parts = skill_content.split('---')
    if len(parts) < 3:
        errors.append("CRITICAL: Missing frontmatter delimiters")
        return errors, warnings
    
    frontmatter_raw = parts[1].strip()
    body = '---'.join(parts[2:]).strip()
    
    # Parse YAML frontmatter
    frontmatter = {}
    current_key = None
    current_value = []
    
    for line in frontmatter_raw.split('\n'):
        if ':' in line and not line.startswith(' '):
            if current_key:
                frontmatter[current_key] = '\n'.join(current_value).strip()
            key, value = line.split(':', 1)
            current_key = key.strip()
            current_value = [value.strip()]
        elif current_key:
            current_value.append(line)
    
    if current_key:
        frontmatter[current_key] = '\n'.join(current_value).strip()
    
    # Required fields check (6 required — tags is OPTIONAL)
    required = ["name", "description", "allowed-tools", "version", "license", "author"]
    for field in required:
        if field not in frontmatter:
            errors.append(f"CRITICAL: Missing required field '{field}'")
    
    # Optional fields reference (no error if missing)
    optional_fields = [
        "model", "context", "agent", "user-invocable", "argument-hint",
        "hooks", "compatibility", "compatible-with", "tags"
    ]
    
    # Name validation
    if "name" in frontmatter:
        name = frontmatter["name"]
        if not re.match(r'^[a-z0-9-]+$', name):
            errors.append(f"CRITICAL: Name '{name}' must be kebab-case")
        if len(name) > 64:
            errors.append(f"CRITICAL: Name '{name}' exceeds 64 characters")
        if "claude" in name or "anthropic" in name:
            errors.append(f"CRITICAL: Name cannot contain reserved words")
    
    # allowed-tools validation — Bash must always be scoped
    if "allowed-tools" in frontmatter:
        tools = frontmatter["allowed-tools"]
        if not tools.startswith('"'):
            errors.append("CRITICAL: allowed-tools must be quoted CSV string")
        # Bash must be scoped: Bash(git:*), Bash(npm:*), etc. — never bare Bash
        if re.search(r'\bBash\b(?!\()', tools):
            errors.append("CRITICAL: Bash must be scoped (e.g., Bash(git:*), Bash(npm:*)) — never bare Bash")
    
    # Version validation
    if "version" in frontmatter:
        version = frontmatter["version"]
        if not re.match(r'^\d+\.\d+\.\d+$', version):
            errors.append(f"CRITICAL: Version '{version}' must be SemVer (x.y.z)")
    
    # Model validation (optional field)
    if "model" in frontmatter:
        model = frontmatter["model"]
        if model not in ("sonnet", "haiku", "opus"):
            errors.append(f"CRITICAL: Model '{model}' must be sonnet, haiku, or opus")
    
    # Description validation
    if "description" in frontmatter:
        desc = frontmatter["description"]
        if "use when" not in desc.lower():
            warnings.append("HIGH: Description should include 'Use when...' phrase")
        if "trigger" not in desc.lower():
            warnings.append("HIGH: Description should include 'Triggers:' with phrases")
    
    # Purpose statement: first paragraph after # Title must be ≤400 chars
    body_lines = body.strip().split('\n')
    if body_lines:
        # Find the first non-empty line after the title
        purpose_lines = []
        found_title = False
        for line in body_lines:
            if line.startswith('# '):
                found_title = True
                continue
            if found_title:
                if line.strip() == '':
                    if purpose_lines:
                        break
                    continue
                if line.startswith('## '):
                    break
                purpose_lines.append(line.strip())
        
        purpose = ' '.join(purpose_lines)
        if len(purpose) > 400:
            errors.append(f"HIGH: Purpose statement is {len(purpose)} chars (max 400)")
        elif not purpose:
            warnings.append("HIGH: Missing purpose statement (1-2 sentences after # Title)")
    
    # Body size validation — ≤150 lines target, 500 max
    lines = body.split('\n')
    words = body.split()
    
    if len(lines) > 500:
        errors.append(f"CRITICAL: Body has {len(lines)} lines (max 500 — move content to references/)")
    elif len(lines) > 150:
        warnings.append(f"MEDIUM: Body has {len(lines)} lines (target ≤150 — consider PDA with references/)")
    
    # Body structure check — 7 required sections
    required_sections = [
        "Overview", "Prerequisites", "Instructions", "Output",
        "Error Handling", "Examples", "Resources"
    ]
    for section in required_sections:
        if f"## {section}" not in body:
            warnings.append(f"MEDIUM: Missing required section '## {section}'")
    
    # Check for ${CLAUDE_PLUGIN_ROOT} (should be ${CLAUDE_SKILL_DIR})
    if "${CLAUDE_PLUGIN_ROOT}" in body:
        errors.append("HIGH: Use ${CLAUDE_SKILL_DIR} instead of ${CLAUDE_PLUGIN_ROOT}")
    
    return errors, warnings

# Validate our skill
errors, warnings = validate_skill_complete(complete_skill)

print("SKILL STANDARD VALIDATION RESULTS")
print("=" * 60)

if not errors and not warnings:
    print("✅ PERFECT! Skill passes all standards!")
    print("\nReady for production deployment.")
else:
    if errors:
        print("\n❌ ERRORS (must fix):")
        for error in errors:
            print(f"   {error}")
    
    if warnings:
        print("\n⚠️  WARNINGS (should fix):")
        for warning in warnings:
            print(f"   {warning}")

print("\n" + "=" * 60)

# Show what is validated
print("\nValidation Coverage:")
print("  ✅ Frontmatter structure")
print("  ✅ Required fields (6 fields — tags is optional)")
print("  ✅ Naming convention (kebab-case)")
print("  ✅ Tool format (quoted CSV, Bash scoping)")
print("  ✅ Version format (SemVer)")
print("  ✅ Model values (sonnet, haiku, opus)")
print("  ✅ Description triggers")
print("  ✅ Purpose statement (≤400 chars)")
print("  ✅ Body size (≤150 target, 500 max)")
print("  ✅ 7 required body sections")
print("  ✅ Path variable (${CLAUDE_SKILL_DIR})")

---

## Step 6: Save Your Skill

Now save the skill to use it!

In [ ]:
import os

def save_skill(skill_content, skill_name, location="project"):
    """Save skill to appropriate location"""
    
    # Determine save path
    if location == "personal":
        skill_dir = os.path.expanduser(f"~/.claude/skills/{skill_name}")
    else:  # project
        skill_dir = f".claude/skills/{skill_name}"
    
    skill_path = f"{skill_dir}/SKILL.md"
    
    # Create directory
    os.makedirs(skill_dir, exist_ok=True)
    
    # Write skill file
    with open(skill_path, 'w') as f:
        f.write(skill_content)
    
    return skill_path

# Example: Save to project (commented out - uncomment to actually save)
# skill_path = save_skill(complete_skill, "test-file-generator", location="project")
# print(f"✅ Skill saved to: {skill_path}")

# Show what would happen
print("TO SAVE YOUR SKILL:")
print("=" * 60)
print("\nUncomment the lines above, or manually:")
print("\n1. Create directory:")
print("   mkdir -p .claude/skills/test-file-generator")
print("\n2. Save SKILL.md:")
print("   # Copy the complete_skill content to:")
print("   .claude/skills/test-file-generator/SKILL.md")
print("\n3. Restart Claude Code (or it will auto-discover)")
print("\n4. Use it:")
print('   User: "Generate tests for src/myfile.py"')
print("   Claude: [Automatically invokes test-file-generator skill]")
print("\n" + "=" * 60)

---

## Step 7: Test Your Skill

How to verify your skill works:

In [ ]:
# Testing checklist
testing_steps = [
    {
        "step": "1. Verify Discovery",
        "command": "Check Claude recognizes the skill",
        "test": "Type: 'List available skills' - should show test-file-generator"
    },
    {
        "step": "2. Test Automatic Invocation",
        "command": "Use trigger phrases",
        "test": "Type: 'Generate tests for myfile.py' - should auto-invoke skill"
    },
    {
        "step": "3. Test Manual Invocation",
        "command": "Use slash command",
        "test": "Type: '/test-file-generator myfile.py'"
    },
    {
        "step": "4. Verify Output",
        "command": "Check generated test file",
        "test": "Verify test file created with correct boilerplate"
    },
    {
        "step": "5. Test Error Handling",
        "command": "Try with non-existent file",
        "test": "Should return helpful error message"
    }
]

print("SKILL TESTING CHECKLIST")
print("=" * 60)
for test in testing_steps:
    print(f"\n{test['step']}")
    print(f"  Command: {test['command']}")
    print(f"  Test: {test['test']}")

print("\n" + "=" * 60)
print("\n💡 Pro tip: Test with real files in a test project first!")

---

## 🎯 Your Turn: Build a Different Skill

Now that you've seen the complete process, build your own!

**Skill Ideas**:
- `readme-generator` - Generate README.md for projects
- `code-reviewer` - Review code for issues
- `api-documenter` - Generate API documentation
- `commit-message-helper` - Generate conventional commits
- `dependency-updater` - Update package dependencies

**Use the builder function**:

In [ ]:
# TODO: Build your own skill!
#
# 1. Choose a task you do frequently
# 2. Plan it (what/when/inputs/outputs/tools)
# 3. Create frontmatter with create_skill_frontmatter()
# 4. Write body with 7 required sections:
#    Overview, Prerequisites, Instructions, Output,
#    Error Handling, Examples, Resources
# 5. Combine and validate
# 6. Save to .claude/skills/your-skill-name/SKILL.md
# 7. Test it!

print("YOUR SKILL GOES HERE!")
print("=" * 60)
print("\nModify this cell to build your own skill.")
print("\nRemember:")
print("  - Use kebab-case names")
print("  - Include trigger phrases in description")
print("  - Scope Bash tools: Bash(git:*), Bash(npm:*) — never bare Bash")
print("  - Use ${CLAUDE_SKILL_DIR} for file references (not ${CLAUDE_PLUGIN_ROOT})")
print("  - Keep body ≤150 lines (500 max) — use references/ for heavy content (PDA)")
print("  - Include all 7 required sections")
print("  - Add a 1-2 sentence purpose statement after # Title (≤400 chars)")
print("  - Validate before saving")
print("\nHappy building!")

---

## Key Takeaways

### Building Process

1. **Plan First**: What/When/Inputs/Outputs/Tools
2. **Write Frontmatter**: 6 required fields (name, description, allowed-tools, version, author, license) + optional fields (tags, model, etc.)
3. **Write Body**: 7 required sections (Overview, Prerequisites, Instructions, Output, Error Handling, Examples, Resources)
4. **Validate**: Intent Solutions standard
5. **Save**: .claude/skills/skill-name/SKILL.md
6. **Test**: Verify discovery and execution

### Best Practices

- **One skill = One clear purpose**
- **Purpose statement**: 1-2 sentences after # Title, max 400 chars
- **Clear triggers** in description for auto-invocation
- **Minimum tools** needed, always scope Bash: `Bash(git:*)`, `Bash(npm:*)`
- **Path variable**: Use `${CLAUDE_SKILL_DIR}` for portable file references
- **Keep it focused**: target 150 lines (500 max) — use Progressive Disclosure Architecture (PDA) with `references/` for heavy content
- **Test thoroughly** before sharing

### Common Mistakes to Avoid

- Using camelCase or snake_case for name (must be kebab-case)
- allowed-tools as array instead of quoted CSV
- Bare `Bash` instead of scoped `Bash(npm:*)` or `Bash(git:*)`
- Missing "Use when..." in description
- No trigger phrases in description
- Body too long without using PDA (`references/implementation.md`)
- Using `${CLAUDE_PLUGIN_ROOT}` instead of `${CLAUDE_SKILL_DIR}`
- Missing required body sections (Overview, Prerequisites, Instructions, Output, Error Handling, Examples, Resources)

---

## Next Steps

**You can now build production skills!**

1. **[04-advanced-patterns.ipynb](04-advanced-patterns.ipynb)** - Multi-phase workflows, sub-agents
2. **[05-skill-standards.ipynb](05-skill-standards.ipynb)** - Complete validation system
3. **Build more skills!** - Practice makes perfect

---

*Intent Solutions Standard Compliant - Version 1.0.0 - MIT License*